In [ ]:
"""Day 23 / Session 2 - Lab Assignment 2: Advanced VAE Knobs - beta-VAE, Per-dim KL, Posterior Collapse & Annealing."""
import numpy as np


def _kl_per_sample(mu, logvar):
    mu = np.asarray(mu, dtype=float)
    logvar = np.asarray(logvar, dtype=float)
    return 0.5 * np.sum(np.exp(logvar) + mu ** 2 - 1 - logvar, axis=-1)


def per_dim_kl(mu, logvar):
    mu = np.asarray(mu, dtype=float)
    logvar = np.asarray(logvar, dtype=float)
    return np.mean(0.5 * (np.exp(logvar) + mu ** 2 - 1 - logvar), axis=0)     # (D,), averaged over the batch


def active_units(mu, logvar, threshold=0.01):
    return int(np.sum(per_dim_kl(mu, logvar) > threshold))                     # strictly greater than the threshold


def beta_vae_loss(recon_log_lik, mu, logvar, beta):
    kl = _kl_per_sample(mu, logvar)
    return float(np.mean(beta * kl - np.asarray(recon_log_lik, dtype=float)))


def kl_annealing_weight(step, warmup_steps):
    if warmup_steps <= 0:
        return 1.0
    return float(min(1.0, max(0.0, step / warmup_steps)))                      # linear warm-up, clamped to [0, 1]
